# **Build the fist MIL pipeline : 1D CNN + Classifier**

In [37]:
from pathlib import Path

CHECKPOINT_DIR = Path("checkpoints_mil")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier des checkpoints :", CHECKPOINT_DIR.resolve())

Dossier des checkpoints : C:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\Base_line\checkpoints_mil


In [38]:
import torch.nn as nn
import torch.optim as optim
import numpy as np
import torchvision.transforms as transforms
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
import torch

In [39]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; refusing to train on CPU.")
print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


In [40]:
from Data_pre_processing.data_cleaning import data_cleaning
import torch
from torch.utils.data import Dataset
from Data_pre_processing.Signal_treatement import split_signals_and_build

## **Building the dataset**

In [41]:
class PCGDataset(Dataset):
    def __init__(self, bags, labels):
        self.bags = (bags)
        self.labels = torch.as_tensor(labels, dtype=torch.float32)
        assert len(bags) == len(labels)

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):

        bag = torch.as_tensor(
            self.bags[idx], dtype=torch.float32
        )
        label = self.labels[idx]

        return bag, label

## **Building the pipeline**

In [ ]:
class Mil_classifier(nn.Module) : 
    def __init__(self):
        # signal = [[],[],[]] = 1
        super(Mil_classifier, self).__init__()
        self.conv1 = nn.Conv1d(1, 10, 5)
        self.pool1 = nn.MaxPool1d(10, 3)
        
        self.conv2 = nn.Conv1d(10, 30, 5)
        self.pool2 = nn.MaxPool1d(15, 6)
        
        self.conv3 = nn.Conv1d(30, 20, 5)
        self.pool3 = nn.MaxPool1d(15, 6)
        
        self.gap = nn.AdaptiveAvgPool1d(1)
        

        self.fc1 = nn.Linear(20, 10)
        self.fc2 = nn.Linear(10,1)

    def forward(self, x : torch.tensor):

        x = self.conv1(x)
        x = self.pool1(F.relu(x))

        x = self.conv2(x)
        x = self.pool2(F.relu(x))

        x = self.conv3(x)
        x = self.pool3(F.relu(x))

        x = self.gap(x).squeeze(-1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        x = x.squeeze(-1)

        x = x.max(dim=0).values
        return x

## **Preparing the data**

In [43]:
counts, data = data_cleaning()

In [44]:
data_splited = split_signals_and_build(data=data)

In [45]:
signals_dict = data_splited.groupby('file')['signal'].apply(lambda x: np.array(x.tolist())).to_dict()
dataset_x = (list(signals_dict.values()))
dataset_y = np.int32(data.iloc[:,1].values)
dataset_y = (dataset_y > 0).astype(np.int64)

In [46]:
dataset = PCGDataset(dataset_x, dataset_y)

## **Training**

In [47]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [48]:
NUM_EPOCHS = 2
LEARNING_RATE = 1e-3
fold_results = []

In [49]:
for fold, (train_idx, valid_idx) in enumerate(
    skf.split(np.zeros(len(dataset_y)), dataset_y),
) : 
    print(f"\n{'=' * 15} Fold {fold} {'=' * 15}")
    torch.manual_seed(100 + fold)

    model = Mil_classifier().to(device)
    
    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-3
    )

    checkpoint_path = CHECKPOINT_DIR / f"fold_{fold}.pt"

    start_epoch = 0

    if checkpoint_path.exists():
        checkpoint = torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=True
        )

        if checkpoint["fold"] != fold:
            raise ValueError("Le checkpoint ne correspond pas au fold.")

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        start_epoch = checkpoint["epoch"]

        print(
            f"Reprise du fold {fold} : "
            f"dernière époque terminée = {start_epoch}"
        )
    else:
        print(f"Nouvel entraînement pour le fold {fold}.")

    train_dataset = Subset(dataset, train_idx.tolist())
    valid_dataset = Subset(dataset, valid_idx.tolist())

    # One bag per batch because bags have variable sizes.
    train_loader = DataLoader(
            train_dataset,
            batch_size=1,
            shuffle=True,
        )

    valid_loader = DataLoader(
            valid_dataset,
            batch_size=1,
            shuffle=False
        )

    for epoch in range(start_epoch, NUM_EPOCHS):
        logits=[]
        model.train()

        total_loss = 0.0
        num_bags = 0

        for bag, label in train_loader:
            # Remove the outer DataLoader batch dimension.
            # bag: [1, num_segments, 1, segment_length]
            bag = bag.squeeze(0).unsqueeze(1).to(device)
            label = label.to(device).reshape(())

            optimizer.zero_grad()

            # One logit for this entire recording.
            logit = model(bag)
            loss = criterion(
                logit.reshape(1),
                label.reshape(1)
            )

            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            num_bags += 1

        mean_loss = total_loss / num_bags

        print(
            f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} "
            f"| Train loss: {mean_loss:.4f}"
        )

        checkpoint = {
            "fold": fold,
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": mean_loss,
            "num_epochs_target": NUM_EPOCHS
        }

        torch.save(checkpoint, checkpoint_path)

        print(f"Checkpoint sauvegardé : {checkpoint_path}")


    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():
        for bag, label in valid_loader : 
            bag = bag.squeeze(0).unsqueeze(1).to(device)

            logit = model(bag)

            proba = torch.sigmoid(logit).item()

            all_probs.append(proba)
            all_labels.append(int(label.item()))
    
    y_true = np.asarray(all_labels, dtype=np.int64)
    y_prob = np.asarray(all_probs)

    y_pred = (y_prob >= 0.5).astype(np.int64)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "sensitivity": recall_score(
            y_true, y_pred, pos_label=1, zero_division=0
        ),
        "f1": f1_score(
            y_true, y_pred, pos_label=1, zero_division=0
        )
        }
    fold_results.append(metrics)
    print(f"Validation bags: {len(y_true)}")

    for name, value in metrics.items():
        print(f"{name}: {value:.4f}")


=============== Fold 0 ===============
Nouvel entraînement pour le fold 0.
Epoch 01/2 | Train loss: 0.6011
Checkpoint sauvegardé : checkpoints_mil\fold_0.pt
Epoch 02/2 | Train loss: 0.5091
Checkpoint sauvegardé : checkpoints_mil\fold_0.pt
Validation bags: 575
accuracy: 0.8000
balanced_accuracy: 0.5000
sensitivity: 0.0000
f1: 0.0000

=============== Fold 1 ===============
Nouvel entraînement pour le fold 1.
Epoch 01/2 | Train loss: 0.6253
Checkpoint sauvegardé : checkpoints_mil\fold_1.pt
Epoch 02/2 | Train loss: 0.5246
Checkpoint sauvegardé : checkpoints_mil\fold_1.pt
Validation bags: 575
accuracy: 0.8000
balanced_accuracy: 0.5000
sensitivity: 0.0000
f1: 0.0000

=============== Fold 2 ===============
Nouvel entraînement pour le fold 2.
Epoch 01/2 | Train loss: 0.6066
Checkpoint sauvegardé : checkpoints_mil\fold_2.pt
Epoch 02/2 | Train loss: 0.5357
Checkpoint sauvegardé : checkpoints_mil\fold_2.pt
Validation bags: 575
accuracy: 0.8017
balanced_accuracy: 0.5000
sensitivity: 0.0000
f1: 0.

In [50]:
print("\n========== Cross-validation summary ==========")

metric_names = fold_results[0].keys()

for name in metric_names:
    values = np.asarray(
        [result[name] for result in fold_results],
        dtype=float
    )

    # Ignore undefined metrics, e.g. AUC if a fold
    # happens to contain only one class.
    valid_values = values[np.isfinite(values)]

    if len(valid_values) == 0:
        print(f"{name}: undefined in all folds")
        continue

    mean = valid_values.mean()

    # Use sample standard deviation if at least two
    # valid fold results exist.
    std = (
        valid_values.std(ddof=1)
        if len(valid_values) > 1
        else 0.0
    )

    print(f"{name}: {mean:.4f} +/- {std:.4f}")


========== Cross-validation summary ==========
accuracy: 0.8010 +/- 0.0009
balanced_accuracy: 0.5000 +/- 0.0000
sensitivity: 0.0000 +/- 0.0000
f1: 0.0000 +/- 0.0000
